# 02 — Data Preprocessing & Feature Engineering
Notebook ini membangun pipeline preprocessing lengkap:
- Handling missing values
- Feature engineering (fitur turunan baru)
- Encoding & scaling
- SMOTE untuk class imbalance
- Validasi pipeline

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from preprocessing import (
    load_dataset, feature_engineering,
    build_preprocessor, prepare_data,
    get_feature_names_out,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES
)
print('Import berhasil ✓')

## 1. Load & Inspeksi Data

In [ ]:
df_raw = load_dataset('../data/heart.csv')
print(f'Shape raw: {df_raw.shape}')
df_raw.head()

In [ ]:
# Cek tipe data
df_raw.dtypes

## 2. Feature Engineering

In [ ]:
df = feature_engineering(df_raw.copy())
print('Fitur baru yang ditambahkan:')
new_feats = [c for c in df.columns if c not in df_raw.columns]
for f in new_feats:
    print(f'  + {f}')
print()
df[new_feats + ['target']].describe().round(3)

In [ ]:
# Visualisasi fitur engineered baru
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, col in enumerate(new_feats):
    for cls, color in zip([0, 1], ['#2E86AB', '#E84855']):
        axes[i].hist(df[df['target']==cls][col].dropna(), bins=20,
                     alpha=0.6, color=color,
                     label='Low Risk' if cls==0 else 'High Risk',
                     edgecolor='white')
    axes[i].set_title(f'Distribusi {col}', fontweight='bold')
    axes[i].legend(fontsize=9)
    axes[i].spines[['top','right']].set_visible(False)

plt.suptitle('Fitur Engineered Baru', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_engineered_features.png', bbox_inches='tight')
plt.show()

## 3. Train/Test Split

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} sampel')
print(f'Test : {X_test.shape[0]} sampel')
print(f'Train class dist: {pd.Series(y_train).value_counts().to_dict()}')
print(f'Test  class dist: {pd.Series(y_test).value_counts().to_dict()}')

## 4. SMOTE — Handle Class Imbalance

In [ ]:
print('Sebelum SMOTE:')
print(pd.Series(y_train).value_counts())

smote = SMOTE(random_state=42)
# SMOTE diterapkan HANYA pada data train (bukan test!)
# Dalam pipeline lengkap, ini ditangani oleh ImbPipeline
X_train_num = X_train.select_dtypes(include='number').fillna(X_train.median(numeric_only=True))
X_resampled, y_resampled = smote.fit_resample(X_train_num, y_train)

print('Setelah SMOTE:')
print(pd.Series(y_resampled).value_counts())

In [ ]:
# Visualisasi efek SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, y_data, title in zip(
    axes,
    [y_train, y_resampled],
    ['Sebelum SMOTE', 'Setelah SMOTE']
):
    vc = pd.Series(y_data).value_counts()
    ax.bar(['Low Risk', 'High Risk'], vc.values,
           color=['#2E86AB', '#E84855'], alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Jumlah Sampel')
    for j, v in enumerate(vc.values):
        ax.text(j, v + 3, str(v), ha='center', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Efek SMOTE pada Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_smote_effect.png', bbox_inches='tight')
plt.show()

## 5. Build & Test Preprocessor Pipeline

In [ ]:
preprocessor = build_preprocessor()

# Fit & transform data training
X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc  = preprocessor.transform(X_test)

feature_names = get_feature_names_out(preprocessor)
print(f'Shape setelah preprocessing:')
print(f'  X_train : {X_train_proc.shape}')
print(f'  X_test  : {X_test_proc.shape}')
print(f'\nFitur output ({len(feature_names)} total):')
for i, f in enumerate(feature_names, 1):
    print(f'  {i:2d}. {f}')

In [ ]:
# Verifikasi: tidak ada NaN setelah preprocessing
print('NaN di X_train_proc:', np.isnan(X_train_proc).sum())
print('NaN di X_test_proc :', np.isnan(X_test_proc).sum())
print()
print('Range nilai setelah StandardScaler:')
df_proc = pd.DataFrame(X_train_proc, columns=feature_names)
print(df_proc.describe().loc[['min','max']].round(3))

## 6. Visualisasi Distribusi Setelah Scaling

In [ ]:
numeric_idx = list(range(len(NUMERIC_FEATURES) + 2))  # +2 engineered

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flatten()

for i, (idx, name) in enumerate(zip(numeric_idx[:7], feature_names[:7])):
    axes[i].hist(X_train_proc[:, idx], bins=30,
                 color='#534AB7', alpha=0.75, edgecolor='white')
    axes[i].set_title(name, fontsize=10)
    axes[i].set_xlabel('Scaled Value')
    axes[i].spines[['top','right']].set_visible(False)

axes[-1].set_visible(False)
plt.suptitle('Distribusi Fitur Setelah StandardScaler', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_scaled_distributions.png', bbox_inches='tight')
plt.show()

## 7. Ringkasan Preprocessing

In [ ]:
print('═'*55)
print('RINGKASAN PREPROCESSING')
print('═'*55)
print(f'Raw features        : {X.shape[1]}')
print(f'Engineered features : {len(new_feats)}')
print(f'Total features      : {len(feature_names)}')
print(f'Train samples       : {X_train.shape[0]}')
print(f'Test samples        : {X_test.shape[0]}')
print(f'Train after SMOTE   : {X_resampled.shape[0]}')
print()
print('Pipeline steps:')
print('  1. feature_engineering()  → +3 fitur baru')
print('  2. SimpleImputer(median)  → handle missing values')
print('  3. OrdinalEncoder         → encode kategorik')
print('  4. StandardScaler         → normalisasi numerik')
print('  5. SMOTE                  → balance kelas')
print()
print('Lanjut ke 03_modeling.ipynb →')